
# Metadaten_Intake_und_Harmonisierung

## Ziel
- Roh-Metadaten aus verschiedenen Quellen laden (Downloader-JSONs, eigene CSV/JSON).
- Vereinheitlichen (Spaltennamen, Typen, Zeitzonen).
- Mit `labels.csv` (Video-Ebene) mergen.
- **Artefakt:** `metadata_base.parquet` (eine Zeile pro `video_id`) **auf gleicher Ebene wie dieses Notebook**.



## Governance & Guardrails (Metadaten-spezifisch)

**Leakage-Regel:** Keine Metriken verwenden, die **nach** dem Labelzeitpunkt entstehen (z. B. finale `likes`, `views`, `shares`), außer explizit als QC.  
**Zeitzonen-Politik:** Alle Zeitfelder in **UTC** speichern; Visualisierungen ggf. in Europe/Berlin.  
**Quellenpriorität:** `Top/Normal Downloader-JSON > data/metadata/*.json|*.csv` (deterministischer Merge).  
**PII/Compliance:** Nur pseudonymisierte Felder speichern (z. B. `uploader_id` statt Klarnamen), keine sensiblen Daten persistieren.  
**Provenienz:** Für jede Zeile und zentrale Spalten die Herkunft dokumentieren (Quelle/Datei/Stand).



## Zu erstellende Artefakte (neben diesem Notebook)
- `metadata_base.parquet` – harmonisierte Basistabelle.
- `schema_metadata_base.json` – Schema-Vertrag.
- `metadata_provenance.csv` – Herkunft pro `video_id`.
- `leakage_scan.csv` – geflaggte Spaltennamen.
- `missing_video_ids.csv` – Abdeckung des Label-Sets.
- (Optional) `video_frames_manifest.csv` – QC-Hinweis auf Frame-Abdeckung.



## Prozessschritte
1. Setup & Import  
2. Datenquellen laden (Downloader-JSONs, `data/metadata/*`)  
3. Harmonisierung & Typen  
4. Deduplizierung auf Video-Ebene  
4a. **Schema-Vertrag & Validierung**  
5. (Optional) Frame-Manifest (QC)  
6. Labels laden & Merge  
6a. **Provenienz protokollieren**  
7. Mini-Report (Missingness)  
7a. **Leakage-Scanner**  
7b. **Coverage-Check**  
8. Persistenz


## 1. Setup & Import

In [ ]:

from pathlib import Path
import pandas as pd, numpy as np, json, re, warnings
warnings.filterwarnings("ignore")

# Projektpfade
PROJECT_ROOT = Path("/mnt/data/viralytics-projekt").resolve()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"  # Artefakte hier speichern
DATA = PROJECT_ROOT / "data"
PROCESSED = DATA / "processed"
FEATURES = PROJECT_ROOT / "features"
METADATA_DIR = DATA / "metadata"
FRAMES_DIR = PROCESSED / "video_frames"  # optional, falls vorhanden

for p in [DATA, PROCESSED, FEATURES, NOTEBOOKS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 160)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Artefakte werden hier gespeichert:", NOTEBOOKS_DIR)


## 2. Helper-Funktionen & Harmonisierung

In [ ]:

def snake(s: str) -> str:
    return re.sub(r"\W+", "_", s.strip()).strip("_").lower()

def cols_to_snake(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [snake(c) for c in df.columns]
    return df

def load_any_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() == ".json":
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
                df = pd.DataFrame(data["data"])
            else:
                df = pd.DataFrame(data if isinstance(data, list) else [data])
        except Exception:
            df = pd.read_json(path, lines=True)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")
    df["source_file"] = str(path)  # Provenienz
    return cols_to_snake(df)

def unify_types(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Standardisierte Zeitfelder
    for c in ["upload_time","create_time","timestamp"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
    # Numeric casting
    for c in ["duration_s","duration","creator_follower_count","creator_posts_count",
              "views","view_count","likes","like_count","comments","comment_count","shares","share_count"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # Bool fields
    for c in ["creator_verified","verified"]:
        if c in df.columns:
            df[c] = df[c].fillna(False).astype(bool)
    # Canonical names
    rename_map = {
        "verified":"creator_verified",
        "like_count":"likes", "view_count":"views", "comment_count":"comments", "share_count":"shares",
        "duration":"duration_s"
    }
    for k,v in rename_map.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k:v})
    # ensure video_id
    for cand in ["video_id","id","aweme_id","tiktok_id"]:
        if cand in df.columns:
            df["video_id"] = df[cand].astype(str)
            break
    return df

def pick_one_row_per_video(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    time_col = "upload_time" if "upload_time" in df.columns else ("create_time" if "create_time" in df.columns else None)
    if time_col:
        df = df.sort_values([ "video_id", time_col], ascending=[True, False])
    else:
        df = df.sort_values(["video_id"])
    return df.drop_duplicates("video_id", keep="first").reset_index(drop=True)


## 3. Datenquellen laden

In [ ]:

from glob import glob

sources = []
# (a) Eigene Metadaten
if METADATA_DIR.exists():
    sources += glob(str(METADATA_DIR / "*.json"))
    sources += glob(str(METADATA_DIR / "*.csv"))

# (b) Downloader-JSONs (Top/Normal)
dl_top = DATA / "Top_100" / "top_metadata.json"
dl_norm = DATA / "Normal_100" / "normal_metadata.json"
for p in [dl_top, dl_norm]:
    if p.exists():
        sources.append(str(p))

print("Gefundene Dateien:", len(sources))
sources[:10]


In [ ]:

tables = []
for s in sources:
    try:
        df = load_any_table(Path(s))
        df = unify_types(df)
        tables.append(df)
    except Exception as e:
        print("⚠️ Fehler beim Laden:", s, e)

assert len(tables) > 0, "Keine Metadaten-Dateien gefunden. Lege Dateien in data/metadata oder nutze die Downloader-JSONs."
meta_raw = pd.concat(tables, ignore_index=True, sort=False)
display(meta_raw.head(3))
print("Shape:", meta_raw.shape)


## 4. Harmonisieren & Deduplizieren (Video-Ebene)

In [ ]:

assert "video_id" in meta_raw.columns, "video_id konnte nicht ermittelt werden – prüfe Quellen/Mapping."
meta_h = cols_to_snake(meta_raw)
meta_h = unify_types(meta_h)

before = meta_h.shape[0]
meta_one = pick_one_row_per_video(meta_h)
after = meta_one.shape[0]
print(f"Dedupliziert: {before} -> {after} Zeilen (eine Zeile pro video_id)")
display(meta_one.head(3))


## 4a. Schema-Vertrag & Validierung

In [ ]:

import json

expected_schema = {
    "required_columns": {
        "video_id": "string",
        "is_viral": "int64",
        "__one_of_time__": ["upload_time","create_time","timestamp"]
    },
    "optional_columns": {
        "creator_verified": "bool",
        "creator_follower_count": "float64",
        "duration_s": "float64",
        "description": "string",
        "title": "string"
    }
}

def normalize_dtype(dt):
    s = str(dt)
    if s.startswith("Int") or s=="int64": return "int64"
    if s.startswith("Float") or s=="float64": return "float64"
    if s in {"bool","boolean"}: return "bool"
    return "string"


## 5. (Optional) Frame-Manifest (QC)

In [ ]:

import re
def build_frame_manifest(frames_dir: Path):
    pat = re.compile(r"id_(\d+)_frame_(\d+)\.(jpg|png)$", re.I)
    rows = []
    if not frames_dir.exists():
        return pd.DataFrame(columns=["video_id","n_frames","sample_frame_path"])
    for p in frames_dir.rglob("*.*"):
        m = pat.search(p.name)
        if not m: 
            continue
        vid = m.group(1)
        idx = int(m.group(2))
        rows.append((vid, idx, str(p)))
    if not rows:
        return pd.DataFrame(columns=["video_id","n_frames","sample_frame_path"])
    dfm = pd.DataFrame(rows, columns=["video_id","frame_idx","path"])
    mani = (dfm.sort_values(["video_id","frame_idx"])
              .groupby("video_id")
              .agg(n_frames=("frame_idx","nunique"),
                   sample_frame_path=("path","first"))
              .reset_index())
    return mani

frames_manifest = build_frame_manifest(FRAMES_DIR)
if not frames_manifest.empty:
    frames_manifest.to_csv(NOTEBOOKS_DIR/"video_frames_manifest.csv", index=False)
    print("Frame-Manifest gespeichert:", NOTEBOOKS_DIR/"video_frames_manifest.csv")
display(frames_manifest.head(3))


## 6. Labels laden & Merge

In [ ]:

labels_path = DATA / "labels.csv"
assert labels_path.exists(), f"labels.csv fehlt unter {labels_path}"
labels = pd.read_csv(labels_path)
labels.columns = [snake(c) for c in labels.columns]
assert {"video_id","is_viral"}.issubset(labels.columns), "labels.csv muss Spalten video_id,is_viral enthalten."

df = labels.merge(meta_one, on="video_id", how="left", validate="one_to_one")
if 'frames_manifest' in globals() and not frames_manifest.empty:
    df = df.merge(frames_manifest, on="video_id", how="left")

display(df.head(3))
print("Shape:", df.shape)


## 6a. Provenienz protokollieren

In [ ]:

source_map = meta_h[["video_id","source_file"]].drop_duplicates("video_id")
prov = df[["video_id"]].merge(source_map, on="video_id", how="left")

prov_path = NOTEBOOKS_DIR / "metadata_provenance.csv"
prov.to_csv(prov_path, index=False)
print("Provenienz gespeichert:", prov_path)
display(prov.head(3))


## 7. Mini-Report (Missingness)

In [ ]:

missing = df.isna().mean().sort_values(ascending=False)
print("Top 20 fehlende Spalten:")
display(missing.head(20))


## 7a. Leakage-Scanner

In [ ]:

import re

LEAKY_PAT = re.compile(r"(like|view|comment|share|play)(_)?(count|rate|s|7d|14d|total)?", re.I)
leaky_cols = [c for c in df.columns if LEAKY_PAT.search(c)]
print("Geflaggte potenziell leaky Spalten:", leaky_cols)

(pd.Series(leaky_cols, name="leaky_columns")
   .to_frame()
   .to_csv(NOTEBOOKS_DIR/"leakage_scan.csv", index=False))
print("Leakage-Report:", NOTEBOOKS_DIR/"leakage_scan.csv")


## 7b. Coverage-Check (Labels vs. Metadaten)

In [ ]:

labels_vids = set(labels["video_id"].astype(str))
meta_vids = set(meta_one["video_id"].astype(str))

missing = sorted(labels_vids - meta_vids)
coverage = 1 - (len(missing) / max(1, len(labels_vids)))
print(f"Coverage: {coverage:.2%}  |  fehlend: {len(missing)} von {len(labels_vids)}")

import pandas as pd
pd.DataFrame({"video_id": missing}).to_csv(NOTEBOOKS_DIR/"missing_video_ids.csv", index=False)
print("Liste fehlender IDs:", NOTEBOOKS_DIR/"missing_video_ids.csv")


### Schema-Validierung jetzt ausführen

In [ ]:

df_for_schema = df.copy()

def normalize_dtype(dt):
    s = str(dt)
    if s.startswith("Int") or s=="int64": return "int64"
    if s.startswith("Float") or s=="float64": return "float64"
    if s in {"bool","boolean"}: return "bool"
    return "string"

if "is_viral" in df_for_schema.columns:
    df_for_schema["is_viral"] = pd.to_numeric(df_for_schema["is_viral"], errors="coerce").fillna(0).astype("int64")
if "creator_verified" in df_for_schema.columns:
    df_for_schema["creator_verified"] = df_for_schema["creator_verified"].fillna(False).astype(bool)
if "video_id" in df_for_schema.columns:
    df_for_schema["video_id"] = df_for_schema["video_id"].astype(str)

expected_schema = {
    "required_columns": {
        "video_id": "string",
        "is_viral": "int64",
        "__one_of_time__": ["upload_time","create_time","timestamp"]
    },
    "optional_columns": {
        "creator_verified": "bool",
        "creator_follower_count": "float64",
        "duration_s": "float64",
        "description": "string",
        "title": "string"
    }
}

missing_req = [c for c in expected_schema["required_columns"] if not c.startswith("__") and c not in df_for_schema.columns]
assert not missing_req, f"Fehlende Pflichtspalten: {missing_req}"

time_field_ok = any(c in df_for_schema.columns for c in expected_schema["required_columns"]["__one_of_time__"])
assert time_field_ok, "Mindestens eines der Zeitfelder fehlt: upload_time | create_time | timestamp"

problems = []
for c, t in expected_schema["required_columns"].items():
    if c.startswith("__"): 
        continue
    if c in df_for_schema.columns:
        got = normalize_dtype(df_for_schema[c].dtype)
        if t != got:
            problems.append((c, t, got))
for c, t in expected_schema["optional_columns"].items():
    if c in df_for_schema.columns:
        got = normalize_dtype(df_for_schema[c].dtype)
        if t != got:
            problems.append((c, t, got))

if problems:
    print("⚠️ Schema-Abweichungen (Spalte, erwartet, ist):")
    for row in problems:
        print("  -", row)

schema_path = NOTEBOOKS_DIR / "schema_metadata_base.json"
schema_path.write_text(json.dumps(expected_schema, indent=2, ensure_ascii=False), encoding="utf-8")
print("Schema geschrieben:", schema_path)


## 8. Persistenz

In [ ]:

out_path = NOTEBOOKS_DIR / "metadata_base.parquet"
df.to_parquet(out_path, index=False)
print("Gespeichert:", out_path, "| Zeilen:", len(df))
